# Lab 5: Adding Passenger Segments

**Does it say "notebook is read only" in a button on your toolbar?  Is the "save" button greyed out?**  
If you want to be able to run this notebook and save your work, you need to copy it to your user directory.
In the file browser on the left (click the folder icon), right click this notebook file (lab-5.ipynb),
copy it, then navigate to your user home directory, and right click and choose paste.  Then open that
copy instead of the read-only copy, and you'll be good to go.

In [ ]:
import passengersim as pax

pax.versions()

In [ ]:
cfg = pax.Config.from_yaml(pax.demo_network("3MKT/DEMO"))
cfg.outputs._write_no_files()
cfg.simulation_controls.num_trials = 1

For this lab, we will be making new customer segments.  To simplify the process, we will 
set our configuration to use common reference prices.  This setup puts the control over
relative willingness-to-pay between segments into the `ChoiceModel` configs, instead of 
having it in the `Demand` configs.  This is less flexible (all demands sharing a choice 
model must have the same reference price multiplier) but also easier to maintain (as 
there is only one value to look at, in the choice model, for scaling the reference prices,
as opposed to unique values for each demand).

In [ ]:
from passengersim.transforms import common_reference_prices

cfg = common_reference_prices(cfg, "leisure")

We can run the demo to establish a baseline result.

In [ ]:
sim = pax.Simulation(cfg)
out = sim.run()

In [ ]:
out.fig_carrier_revenues()

To establish a new customer segment, we need to think about a few things:

- What choice model do these customers use to make their purchase decisions?
- When do these customers arrive during the booking curve?

In [ ]:
cm = sim.choice_models["leisure"]
dmd = sim.demands.select(segment="leisure", orig="BOS", dest="LAX")

In [ ]:
cm.max_wtp(dmd.reference_price, n_draws=1_000_000, raw=True)

One big part of the choice model is the customer's maximum willingness to pay. This is generally analogous to 
a budget for travel: each customer can pay only so much, and not more.  Any product which costs more than
the budget is infeasible for that customer and is immediately removed from consideration, even if it is
otherwise very appealing.

We can visualize the distribution of maximum willingness to pay in any O-D market by passenger segment:

In [ ]:
cfg.fig_max_wtp_distributions(orig="BOS", dest="LAX")

Suppose we want to add a "family" segment, with a somewhat higher willingness to pay than the leisure segment.  
We can add the new segment like this:

In [ ]:
cfg.choice_models["family"] = cfg.choice_models["leisure"].model_copy(deep=True)

cfg.choice_models["family"].reference_price_multiplier = 1.75
cfg.choice_models["family"].name = "family"

We also need to make some demands that can generate customers from the new segment.

In [ ]:
cfg.demands

In [ ]:
leisure_to_LAX = [d for d in cfg.demands if d.segment == "leisure" and d.dest == "LAX"]
leisure_to_LAX

In [ ]:
family_to_LAX = []

for d in leisure_to_LAX:
    f = d.model_copy(deep=True)
    f.segment = "family"
    f.choice_model = "family"
    f.curve = "c_family"
    f.base_demand = 0.25 * d.base_demand  # add new family demand equal to 25% of original leisure demand
    family_to_LAX.append(f)

    d.base_demand = 0.75 * d.base_demand  # reduce existing regular leisure demand by 25%

cfg.demands.extend(family_to_LAX)

In [ ]:
cfg.fig_max_wtp_distributions(orig="BOS", dest="LAX")

We also need to set up the booking curve that tells the simulator when these customer arrive.
We can see our current booking curves like this:

In [ ]:
cfg.fig_booking_curves()

When we created the new "family" demands, we assigned them booking curve "c_family" but we didn't say what that is yet.

In [ ]:
from pytest import raises

with raises(Exception) as e:
    cfg.model_revalidate()

print(e)

We'll need to define that booking curve to run with this config.  Let's do that now.

In [ ]:
cfg.booking_curves["c_family"] = cfg.booking_curves["c2"].model_copy(deep=True)
cfg.booking_curves["c_family"].name = "c_family"
cfg.booking_curves["c_family"].curve = {
    63: 0.33,
    56: 0.51,
    49: 0.72,
    42: 0.79,
    35: 0.81,
    31: 0.83,
    28: 0.86,
    24: 0.88,
    21: 0.90,
    17: 0.91,
    14: 0.93,
    10: 0.95,
    7: 0.97,
    5: 0.98,
    3: 0.99,
    1: 1.0,
}

Now we can review the config with the new booking curve.

In [ ]:
cfg.fig_booking_curves()

Let's run the model again.

In [ ]:
sim2 = pax.MultiSimulation(cfg)
out2 = sim2.run()

In [ ]:
from passengersim.contrast import Contrast

comp = Contrast({"original": out, "with_family": out2})

In [ ]:
comp.fig_carrier_revenues()

In [ ]:
comp.fig_bookings_by_timeframe(by_carrier="AL1", by_class=True)

In [ ]:
cfg.to_yaml_parts(directory="lab-5-family")

In [ ]:
# let's try an "executive" demand as well

In [ ]:
cfg.demands[4]

In [ ]:
d1 = cfg.demands[4]
e1 = d1.model_copy(deep=True)
e1.segment = "executive"
e1.choice_model = "executive"
e1.curve = "c_exec"
e1.base_demand = 0.50 * d1.base_demand

cfg.demands.append(e1)
d1.base_demand = 0.50 * d1.base_demand

In [ ]:
cfg.choice_models["executive"] = cfg.choice_models["business"].model_copy(deep=True)

cfg.choice_models["executive"].reference_price_multiplier = 4.0
cfg.choice_models["executive"].emult = 1.9
cfg.choice_models["executive"].name = "executive"

In [ ]:
cfg.booking_curves["c_exec"] = cfg.booking_curves["c1"].model_copy(deep=True)
cfg.booking_curves["c_exec"].name = "c_exec"
cfg.booking_curves["c_exec"].curve = {
    63: 0.0,
    56: 0.0,
    49: 0.0,
    42: 0.0,
    35: 0.0,
    31: 0.0,
    28: 0.0,
    24: 0.0,
    21: 0.03,
    17: 0.06,
    14: 0.10,
    10: 0.13,
    7: 0.18,
    5: 0.36,
    3: 0.77,
    1: 1.0,
}

In [ ]:
cfg.fig_booking_curves()

In [ ]:
sim3 = pax.MultiSimulation(cfg)
out3 = sim3.run()

In [ ]:
comp = Contrast({"original": out, "with_family": out2, "with_executive": out3})

In [ ]:
comp.fig_carrier_revenues()

In [ ]:
comp.fig_bookings_by_timeframe(by_carrier="AL1", by_class=True)